# Milestone 3 - Modelação e Avaliação (Objetivo 2)

Neste notebook é desenvolvido um índice de risco de abandono (`Attrition`), com base nas probabilidades previstas por modelos de classificação supervisionada. Numa fase inicial, foram testados e comparados diferentes algoritmos preditivos, tendo sido selecionada a **Regressão Logística** como modelo final, devido ao seu equilíbrio entre desempenho, estabilidade e interpretabilidade. Posteriormente, o modelo foi otimizado através de *tuning* de hiperparâmetros, permitindo melhorar a sua capacidade de identificação de colaboradores em risco e garantir uma utilização eficaz em contexto de negócio.

Esta etapa enquadra-se nas fases de *Modelling* e *Evaluation* da metodologia CRISP-DM, tendo como objetivo transformar previsões probabilísticas em **informação acionável para suporte à decisão em contexto de negócio**.

Ao contrário de uma abordagem puramente classificativa, este notebook foca-se na **utilização das probabilidades previstas**, permitindo uma análise mais granular do risco individual de cada colaborador.

---

## Estrutura da Análise

Foram consideradas diferentes versões do modelo de Regressão Logística:

- **Baseline:** Regressão Logística 
- **Candidato 1:** Naive Bayes
- **Candidato 2:** LDA (Linear Discriminant Analysis)
- **Candidato 3:** KNN (K-Nearest Neighbors)
- **Candidato 4:** Extra Trees
- **Candidato 5:** SVM (Support Vector Machine)
- **Candidato 6:** ADABOOST
- **Candidato 7:** LIGHTGBM (Light Gradient Boosting Machine)
- **Candidato 8:** XGBOOST
- **Candidato 9:** Random Forest
---

## Metodologia

A análise segue os seguintes passos principais:

- Preparação e normalização dos dados (StandardScaler), apenas no Baseline (Regressão Logística), KNN e SVM.
- Divisão do dataset em treino e teste (80/20)
- Treino dos modelos e avaliação do desempenho
- Escolha do modelo com melhor desempenho
- Otimização dos hiperparâmetros através de validação cruzada estratificada

- **retificar daqui para baixo**

- Avaliação do modelo otimizado em dados de teste
- **Otimização do threshold de decisão**, com base na maximização do F1-score
- Comparação entre baseline, modelo otimizado e modelo com threshold ajustado
- Geração das probabilidades de saída para todos os colaboradores;
- Construção de um índice de risco categórico
- Análise da distribuição das probabilidades e dos níveis de risco
- Caracterização dos perfis associados a cada categoria de risco
- Avaliação da coerência entre o índice de risco e a variável real (*Attrition*)
- Interpretação dos resultados numa perspetiva de negócio

---

## Definição do Índice de Risco

Com base nas probabilidades previstas, os colaboradores são classificados nas seguintes categorias:

- **Risco Baixo:** probabilidade < 30%  
- **Risco Médio:** 30% ≤ probabilidade < 50%  
- **Risco Alto:** 50% ≤ probabilidade < 70%  
- **Risco Crítico:** probabilidade ≥ 70%  

Esta segmentação permite transformar uma variável contínua em **grupos interpretáveis e acionáveis**, facilitando a priorização de intervenções por parte da gestão de recursos humanos.

---

## Métricas de Avaliação

O desempenho dos modelos é avaliado com base em métricas adequadas a problemas de classificação com classes desbalanceadas:

- **F1-Score** → métrica principal (equilíbrio entre precision e recall);  
- **Recall** → capacidade de identificar colaboradores em risco de saída;  
- **Precision** → controlo de falsos positivos;  
- **AUC-ROC** → capacidade discriminativa global do modelo.

---

## Objetivo Final

Construir um índice de risco robusto e interpretável que permita:

- Identificar colaboradores com maior probabilidade de saída
- Apoiar decisões estratégicas de retenção de talento
- Priorizar intervenções com base no nível de risco
- Traduzir resultados analíticos em **insights de negócio acionáveis**.

---

**Autores:** Luís Figueira, Martim Ferreira e Mateus Afonso

# Índice de Risco

In [ ]:
# 1. IMPORTAÇÕES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, RocCurveDisplay,
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings("ignore")
print("Bibliotecas importadas com sucesso.")
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 2. CARREGAMENTO DO DATASET

#url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
#df = pd.read_csv(url)
#print(f"Dataset carregado: {df.shape[0]} linhas, {df.shape[1]} colunas")
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 3. PREPARAÇÃO DAS FEATURES

#cols_remover = ["Attrition", "OverTime", "Gender",
#                "BusinessTravel", "Department", "EducationField",
#                "JobRole", "MaritalStatus"]

#cols_remover = [c for c in cols_remover if c in df.columns]
#df_model = df.drop(columns=cols_remover)

#TARGET = "Attrition_bin"
#X = df_model.drop(columns=[TARGET])
#y = df_model[TARGET]
#X = X.select_dtypes(include=[np.number])

#print(f"Features utilizadas: {X.shape[1]}")
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 4. DIVISÃO TREINO / TESTE — Gerar
#import os
#import zipfile
#from IPython.display import FileLink, display

#treino_path = "data/processed/Objetivo2/treino"
#teste_path  = "data/processed/Objetivo2/teste"

# 80% treino, 20% teste
# stratify=y — garante a mesma proporção de Yes/No em treino e teste
#X_train, X_test, y_train, y_test = train_test_split(
#    X, y, test_size=0.2, random_state=42, stratify=y
#)

# Criar as pastas e guardar os splits
#os.makedirs(treino_path, exist_ok=True)
#os.makedirs(teste_path, exist_ok=True)
#X_train.to_csv(f"{treino_path}/X_train.csv", index=False)
#y_train.to_csv(f"{treino_path}/y_train.csv", index=False)
#X_test.to_csv(f"{teste_path}/X_test.csv", index=False)
#y_test.to_csv(f"{teste_path}/y_test.csv", index=False)

# Criar ZIP com a estrutura de pastas completa
#zip_path = "data/processed/Objetivo2/splits.zip"
#with zipfile.ZipFile(zip_path, "w") as zipf:
#    zipf.write(f"{treino_path}/X_train.csv", "treino/X_train.csv")
#    zipf.write(f"{treino_path}/y_train.csv", "treino/y_train.csv")
#    zipf.write(f"{teste_path}/X_test.csv",   "teste/X_test.csv")
#    zipf.write(f"{teste_path}/y_test.csv",   "teste/y_test.csv")

#print("Splits gerados! :")
#display(FileLink(zip_path))
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 4. DIVISÃO TREINO / TESTE — CARREGAR DO GITHUB
base_treino = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/Objetivo2/treino"
base_teste  = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/Objetivo2/teste"

# Carregar diretamente do GitHub
X_train = pd.read_csv(f"{base_treino}/X_train.csv")
y_train = pd.read_csv(f"{base_treino}/y_train.csv").squeeze()
X_test  = pd.read_csv(f"{base_teste}/X_test.csv")
y_test  = pd.read_csv(f"{base_teste}/y_test.csv").squeeze()

print(f"Treino: {X_train.shape[0]} obs | Teste: {X_test.shape[0]} obs")
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

## 0. Baseline - Regressão Logística 

In [ ]:
# 5. TREINO — REGRESSÃO LOGÍSTICA (BASELINE)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression())
])

pipeline.fit(X_train, y_train)
print("Modelo treinado.")
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 6. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }
    # Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 6.1 TREINO
r_treino = avaliar_modelo(pipeline, X_train, y_train, "Treino")
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 6.2 TESTE
r_teste = avaliar_modelo(pipeline, X_test, y_test, "Teste")
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 7. COMPARAÇÃO TREINO vs TESTE

print("===== COMPARAÇÃO TREINO vs TESTE =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*42}")
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = r_treino[metrica]
    val_teste  = r_teste[metrica]
    diff       = val_treino - val_teste
    nome       = metrica.upper() if metrica != "acc" else "Accuracy"
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

diff_f1 = r_treino["f1"] - r_teste["f1"]
if diff_f1 > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting.")
    # Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 8. CURVA ROC — TREINO E TESTE

fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(
    r_treino["y"], r_treino["y_proba"],
    name=f"Treino (AUC={r_treino['auc']:.3f})", ax=ax, color="steelblue"
)
RocCurveDisplay.from_predictions(
    r_teste["y"], r_teste["y_proba"],
    name=f"Teste  (AUC={r_teste['auc']:.3f})", ax=ax, color="darkorange"
)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("Curva ROC — Regressão Logística (Baseline)")
plt.tight_layout()
plt.savefig("roc_treino_vs_teste.png", dpi=150, bbox_inches="tight")
plt.show()
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 9. GERAR PROBABILIDADES DE SAÍDA (dataset completo)
url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features que fizeste no início
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"])
X_completo = X_completo.select_dtypes(include=[np.number])

df_risco = df_completo.copy()
df_risco["prob_saida"] = pipeline.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco)} colaboradores.")
print(df_risco["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 10. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

df_risco["nivel_risco"] = df_risco["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem = df_risco["nivel_risco"].value_counts()
percentagem = df_risco["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n = contagem.get(cat, 0)
    pct = percentagem.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 11. ANÁLISE POR CATEGORIA DE RISCO

cols_analise = ["prob_saida", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
            "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"]:
    if col in df_risco.columns:
        cols_analise.append(col)

print("\n===== PERFIL MEDIO POR CATEGORIA DE RISCO =====")
perfil = df_risco.groupby("nivel_risco")[cols_analise].mean().reindex(ORDEM).round(3)
display(perfil)
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 12. TOP 20 COLABORADORES COM MAIOR RISCO

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco.columns:
        cols_top.append(col)

top20 = df_risco.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAIDA =====")
display(top20)
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 13. VISUALIZAÇÕES DO ÍNDICE DE RISCO

# 13.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Novos thresholds
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo (<30%)")
ax.axvline(0.50, color="gold", linestyle="--", linewidth=1.5, label="Médio (30–50%)")
ax.axvline(0.70, color="red", linestyle="--", linewidth=1.5, label="Alto (50–70%) / Crítico (>70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída")

ax.legend()
plt.tight_layout()
plt.savefig("distribuicao_probabilidades.png", dpi=150, bbox_inches="tight")
plt.show()
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 13.2 Contagem por categoria (ATUALIZADO)

# Nova ordem e cores coerentes com risco
ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",     # verde
    "Médio": "#f39c12",     # laranja
    "Alto": "#e74c3c",      # vermelho
    "Crítico": "#8e0000"    # vermelho escuro
}

fig, ax = plt.subplots(figsize=(9, 5))

vals = [contagem.get(c, 0) for c in ORDEM]
bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

# Labels com contagem + %
for bar, cat in zip(bars, ORDEM):
    n = contagem.get(cat, 0)
    pct = percentagem.get(cat, 0.0)

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição de Colaboradores por Categoria de Risco")

plt.tight_layout()
plt.savefig("categorias_risco.png", dpi=150, bbox_inches="tight")
plt.show()
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 13.3 Boxplot por categoria
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df_risco, x="nivel_risco", y="prob_saida", order=ORDEM, ax=ax)
ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saida")
ax.set_title("Distribuicao das Probabilidades por Categoria")
plt.tight_layout()
plt.savefig("boxplot_risco.png", dpi=150, bbox_inches="tight")
plt.show()
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

In [ ]:
# 14. RESUMO FINAL

print("=" * 55)
print("RESUMO — BASELINE")
print("=" * 55)

print(f"  Modelo:        Regressão Logística (parâmetros default)")
print(f"  Colaboradores: {len(df_risco)}")

print(f"\n  {'Métrica':<12} {'Treino':>8} {'Teste':>8}")
print("-" * 32)
for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12} {r_treino[metrica]:>8.4f} {r_teste[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")
print(f"  Baixo:    prob < 30%        -> {contagem.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem.get('Crítico', 0)} colaboradores")

print("=" * 55)
# Autores: Figueira, L., Afonso, M. e Ferreira, M.

#  Treino e Avaliação Comparativa do Desempenho de Modelos Candidatos


## 1. Candidato - Naive Bayes


In [ ]:
# 15. TREINO — NAIVE BAYES
pipeline_nb = Pipeline([("nb", GaussianNB())])

pipeline_nb.fit(X_train, y_train)
print("Modelo Naive Bayes treinado.")

In [ ]:
# 16. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE (NAIVE BAYES)

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }

In [ ]:
# 16.1 TREINO
resultados_treino_nb = avaliar_modelo(pipeline_nb, X_train, y_train, "Treino")

In [ ]:
# 16.2 TESTE
resultados_teste_nb  = avaliar_modelo(pipeline_nb, X_test,  y_test,  "Teste")

In [ ]:
# 17. COMPARAÇÃO TREINO vs TESTE (NAIVE BAYES)

print("===== COMPARAÇÃO TREINO vs TESTE (NAIVE BAYES) =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*44}")

# Iterar sobre as métricas guardadas no dicionário
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino_nb[metrica]
    val_teste  = resultados_teste_nb[metrica]
    diff       = val_treino - val_teste
    
    # Formatação do nome da métrica
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

# Validação automática de Overfitting focada no F1-Score
diff_f1_nb = resultados_treino_nb["f1"] - resultados_teste_nb["f1"]

print() # Linha em branco para limpeza visual
if diff_f1_nb > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting.")

In [ ]:
# 18. CURVAS ROC SOBREPOSTAS (NAIVE BAYES)

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Treino
RocCurveDisplay.from_predictions(
    resultados_treino_nb["y"], resultados_treino_nb["y_proba"],
    name=f"Treino (AUC={resultados_treino_nb['auc']:.3f})", ax=ax, color="steelblue"
)

# Curva de Teste
RocCurveDisplay.from_predictions(
    resultados_teste_nb["y"], resultados_teste_nb["y_proba"],
    name=f"Teste  (AUC={resultados_teste_nb['auc']:.3f})", ax=ax, color="darkorange"
)

# Linha de referência (o "acaso")
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_title("Curva ROC — Naive Bayes (Treino vs Teste)")
plt.tight_layout()
plt.savefig("roc_treino_vs_teste_nb.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 19. GERAR PROBABILIDADES DE SAÍDA (dataset completo - NAIVE BAYES)

url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

# Adicionei errors='ignore' para evitar erros caso a Attrition_bin não esteja no CSV
X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"], errors='ignore')
X_completo = X_completo.select_dtypes(include=[np.number])

# Usamos df_risco_nb para não sobrescrever o teu trabalho da Regressão Logística
df_risco_nb = df_completo.copy()

# A MUDANÇA PRINCIPAL: Chamar o pipeline do Naive Bayes
df_risco_nb["prob_saida"] = pipeline_nb.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco_nb)} colaboradores (Naive Bayes).")
print(df_risco_nb["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 20. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO (NAIVE BAYES)

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

# Aplicar a função ao DataFrame do Naive Bayes
df_risco_nb["nivel_risco"] = df_risco_nb["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem_nb    = df_risco_nb["nivel_risco"].value_counts()
percentagem_nb = df_risco_nb["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO (NAIVE BAYES) =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n   = contagem_nb.get(cat, 0)
    pct = percentagem_nb.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
# 21. ANÁLISE POR CATEGORIA DE RISCO (NAIVE BAYES)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cols_analise = ["prob_saida", "Attrition_bin"]

# Adicionar variáveis de negócio para caracterizar cada nível de risco
for col in [
    "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
    "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"
]:
    if col in df_risco_nb.columns:
        cols_analise.append(col)

print("\n===== PERFIL MÉDIO POR CATEGORIA DE RISCO (NAIVE BAYES) =====")

perfil_nb = (
    df_risco_nb
    .groupby("nivel_risco")[cols_analise]
    .mean()
    .reindex(ORDEM)
    .round(3)
)

display(perfil_nb)

In [ ]:
# 22. TOP 20 COLABORADORES COM MAIOR RISCO (NAIVE BAYES)

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco_nb.columns:
        cols_top.append(col)

top20_nb = df_risco_nb.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA (NAIVE BAYES) =====")
display(top20_nb)

In [ ]:
# 23. VISUALIZAÇÕES DO ÍNDICE DE RISCO (NAIVE BAYES)

# 23.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco_nb["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Novos thresholds corretos
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo/Médio (30%)")
ax.axvline(0.50, color="red", linestyle="--", linewidth=1.5, label="Médio/Alto (50%)")
ax.axvline(0.70, color="darkred", linestyle="--", linewidth=1.5, label="Alto/Crítico (70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída (Naive Bayes)")

ax.legend()

plt.tight_layout()
plt.savefig("distribuicao_probabilidades_nb.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 23.2 Contagem por categoria (NAIVE BAYES)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",
    "Médio": "#f39c12",
    "Alto": "#e74c3c",
    "Crítico": "#8e0000"
}

fig, ax = plt.subplots(figsize=(9, 5))

vals = [contagem_nb.get(c, 0) for c in ORDEM]
bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

for bar, cat in zip(bars, ORDEM):
    n = contagem_nb.get(cat, 0)
    pct = percentagem_nb.get(cat, 0.0)
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco (Naive Bayes)")

plt.tight_layout()
plt.savefig("categorias_risco_nb.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 23.3 Boxplot por categoria (NAIVE BAYES)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = ["#2ecc71", "#f39c12", "#e74c3c", "#8e0000"]

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df_risco_nb,
    x="nivel_risco",
    y="prob_saida",
    order=ORDEM,
    palette=cores,
    ax=ax
)

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saída")
ax.set_title("Distribuição das Probabilidades por Categoria (Naive Bayes)")

plt.tight_layout()
plt.savefig("boxplot_risco_nb.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 24. RESUMO FINAL (NAIVE BAYES)

print("=" * 55)
print("RESUMO — Naive Bayes")
print("=" * 55)

print(f"  Modelo:         Naive Bayes (parâmetros default)")
print(f"  Colaboradores:  {len(df_risco_nb)}")

print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12}  {resultados_treino_nb[metrica]:>8.4f}  {resultados_teste_nb[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")

print(f"  Baixo:    prob < 30%        -> {contagem_nb.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem_nb.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem_nb.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem_nb.get('Crítico', 0)} colaboradores")

print("=" * 55)

## 2. Candidato  - LDA (Linear Discriminant Analysis)

In [ ]:
# 25. TREINO — LDA (LINEAR DISCRIMINANT ANALYSIS)
pipeline_lda = Pipeline([("lda", LinearDiscriminantAnalysis())])

pipeline_lda.fit(X_train, y_train)
print("Modelo LDA treinado.")

In [ ]:
# 26. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE (LDA)

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }

In [ ]:
# 26.1 TREINO
resultados_treino_lda = avaliar_modelo(pipeline_lda, X_train, y_train, "Treino")

In [ ]:
# 26.2 TESTE 
resultados_teste_lda  = avaliar_modelo(pipeline_lda, X_test,  y_test,  "Teste")

In [ ]:
# 27. COMPARAÇÃO TREINO vs TESTE (LDA)

print("===== COMPARAÇÃO TREINO vs TESTE (LDA) =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*44}")

# Iterar sobre as métricas guardadas no dicionário
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino_lda[metrica]
    val_teste  = resultados_teste_lda[metrica]
    diff       = val_treino - val_teste
    
    # Formatação do nome da métrica
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

# Validação automática de Overfitting focada no F1-Score
diff_f1_lda = resultados_treino_lda["f1"] - resultados_teste_lda["f1"]

print() # Linha em branco para limpeza visual
if diff_f1_lda > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting.")

In [ ]:
# 28. CURVAS ROC SOBREPOSTAS (LDA)
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Treino
RocCurveDisplay.from_predictions(
    resultados_treino_lda["y"], resultados_treino_lda["y_proba"],
    name=f"Treino (AUC={resultados_treino_lda['auc']:.3f})", ax=ax, color="steelblue"
)

# Curva de Teste
RocCurveDisplay.from_predictions(
    resultados_teste_lda["y"], resultados_teste_lda["y_proba"],
    name=f"Teste  (AUC={resultados_teste_lda['auc']:.3f})", ax=ax, color="darkorange"
)

# Linha de referência (o "acaso")
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_title("Curva ROC — LDA (Treino vs Teste)")
plt.tight_layout()
plt.savefig("roc_treino_vs_teste_lda.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 29. GERAR PROBABILIDADES DE SAÍDA (dataset completo - LDA)

url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

# Evitar erros caso a Attrition_bin não esteja no CSV
X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"], errors='ignore')
X_completo = X_completo.select_dtypes(include=[np.number])

# Usamos df_risco_lda para não sobrescrever o teu trabalho anterior
df_risco_lda = df_completo.copy()

# A MUDANÇA PRINCIPAL: Chamar o pipeline do LDA
df_risco_lda["prob_saida"] = pipeline_lda.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco_lda)} colaboradores (LDA).")
print(df_risco_lda["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 30. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO (LDA)

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

# Aplicar a função ao DataFrame do LDA
df_risco_lda["nivel_risco"] = df_risco_lda["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem_lda    = df_risco_lda["nivel_risco"].value_counts()
percentagem_lda = df_risco_lda["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO (LDA) =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n   = contagem_lda.get(cat, 0)
    pct = percentagem_lda.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
# 31. ANÁLISE POR CATEGORIA DE RISCO (LDA)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cols_analise = ["prob_saida", "Attrition_bin"]

# Adicionar variáveis de negócio para caracterizar cada nível de risco
for col in [
    "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
    "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"
]:
    if col in df_risco_lda.columns:
        cols_analise.append(col)

print("\n===== PERFIL MÉDIO POR CATEGORIA DE RISCO (LDA) =====")

perfil_lda = (
    df_risco_lda
    .groupby("nivel_risco")[cols_analise]
    .mean()
    .reindex(ORDEM)
    .round(3)
)

display(perfil_lda)

In [ ]:
# 32. TOP 20 COLABORADORES COM MAIOR RISCO (LDA)

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco_lda.columns:
        cols_top.append(col)

top20_lda = df_risco_lda.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA (LDA) =====")
display(top20_lda)

In [ ]:
# 33. VISUALIZAÇÕES DO ÍNDICE DE RISCO (LDA)

# 33.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco_lda["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Thresholds atualizados
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo/Médio (30%)")
ax.axvline(0.50, color="red", linestyle="--", linewidth=1.5, label="Médio/Alto (50%)")
ax.axvline(0.70, color="darkred", linestyle="--", linewidth=1.5, label="Alto/Crítico (70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída (LDA)")

ax.legend()

plt.tight_layout()
plt.savefig("distribuicao_probabilidades_lda.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 33.2 Contagem por categoria (LDA)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",
    "Médio": "#f39c12",
    "Alto": "#e74c3c",
    "Crítico": "#8e0000"
}

fig, ax = plt.subplots(figsize=(9, 5))

vals = [contagem_lda.get(c, 0) for c in ORDEM]

bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

for bar, cat in zip(bars, ORDEM):
    n   = contagem_lda.get(cat, 0)
    pct = percentagem_lda.get(cat, 0.0)
    
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco (LDA)")

plt.tight_layout()
plt.savefig("categorias_risco_lda.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 33.3 Boxplot por categoria (LDA)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = ["#2ecc71", "#f39c12", "#e74c3c", "#8e0000"]

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df_risco_lda,
    x="nivel_risco",
    y="prob_saida",
    order=ORDEM,
    palette=cores,
    ax=ax
)

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saída")
ax.set_title("Distribuição das Probabilidades por Categoria (LDA)")

plt.tight_layout()
plt.savefig("boxplot_risco_lda.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 34. RESUMO FINAL (LDA)

print("=" * 55)
print("RESUMO — LDA")
print("=" * 55)

print(f"  Modelo:         LDA (parâmetros default)")
print(f"  Colaboradores:  {len(df_risco_lda)}")

print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12}  {resultados_treino_lda[metrica]:>8.4f}  {resultados_teste_lda[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")

print(f"  Baixo:    prob < 30%        -> {contagem_lda.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem_lda.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem_lda.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem_lda.get('Crítico', 0)} colaboradores")

print("=" * 55)

## 3. Candidato  - KNN (K-NEAREST NEIGHBORS)
Modelo KNN com StandardScaler e classificação de risco.

In [ ]:
# 35. TREINO — KNN (K-NEAREST NEIGHBORS)
pipeline_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()) 
])

pipeline_knn.fit(X_train, y_train)
print("Modelo KNN treinado.")

In [ ]:
# 36. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE (KNN)
def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }

In [ ]:
# 36.1 TREINO
resultados_treino_knn = avaliar_modelo(pipeline_knn, X_train, y_train, "Treino")

In [ ]:
# 36.2 TESTE
resultados_teste_knn  = avaliar_modelo(pipeline_knn, X_test,  y_test,  "Teste")

In [ ]:
# 37. COMPARAÇÃO TREINO vs TESTE (KNN)

print("===== COMPARAÇÃO TREINO vs TESTE (KNN) =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*44}")

# Iterar sobre as métricas guardadas no dicionário
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino_knn[metrica]
    val_teste  = resultados_teste_knn[metrica]
    diff       = val_treino - val_teste
    
    # Formatação do nome da métrica
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

# Validação automática de Overfitting focada no F1-Score
diff_f1_knn = resultados_treino_knn["f1"] - resultados_teste_knn["f1"]

print() # Linha em branco para limpeza visual
if diff_f1_knn > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting.")

In [ ]:
# 38. CURVAS ROC SOBREPOSTAS (KNN)
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Treino
RocCurveDisplay.from_predictions(
    resultados_treino_knn["y"], resultados_treino_knn["y_proba"],
    name=f"Treino (AUC={resultados_treino_knn['auc']:.3f})", ax=ax, color="steelblue"
)

# Curva de Teste
RocCurveDisplay.from_predictions(
    resultados_teste_knn["y"], resultados_teste_knn["y_proba"],
    name=f"Teste  (AUC={resultados_teste_knn['auc']:.3f})", ax=ax, color="darkorange"
)

# Linha de referência (o "acaso")
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_title("Curva ROC — KNN (Treino vs Teste)")
plt.tight_layout()
plt.savefig("roc_treino_vs_teste_knn.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 39. GERAR PROBABILIDADES DE SAÍDA (dataset completo - KNN)
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

# Evitar erros caso a Attrition_bin não esteja no CSV
X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"], errors='ignore')
X_completo = X_completo.select_dtypes(include=[np.number])

# Usamos df_risco_knn para não sobrescrever o teu trabalho anterior
df_risco_knn = df_completo.copy()

# A MUDANÇA PRINCIPAL: Chamar o pipeline do KNN
df_risco_knn["prob_saida"] = pipeline_knn.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco_knn)} colaboradores (KNN).")
print(df_risco_knn["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 40. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO (KNN)

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

# Aplicar a função ao DataFrame do KNN
df_risco_knn["nivel_risco"] = df_risco_knn["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem_knn    = df_risco_knn["nivel_risco"].value_counts()
percentagem_knn = df_risco_knn["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO (KNN) =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n   = contagem_knn.get(cat, 0)
    pct = percentagem_knn.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
# 41. ANÁLISE POR CATEGORIA DE RISCO (KNN)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cols_analise = ["prob_saida", "Attrition_bin"]

# Adicionar variáveis de negócio para caracterizar cada nível de risco
for col in [
    "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
    "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"
]:
    if col in df_risco_knn.columns:
        cols_analise.append(col)

print("\n===== PERFIL MÉDIO POR CATEGORIA DE RISCO (KNN) =====")

perfil_knn = (
    df_risco_knn
    .groupby("nivel_risco")[cols_analise]
    .mean()
    .reindex(ORDEM)
    .round(3)
)

display(perfil_knn)

In [ ]:
# 42. TOP 20 COLABORADORES COM MAIOR RISCO (KNN)

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco_knn.columns:
        cols_top.append(col)

top20_knn = df_risco_knn.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA (KNN) =====")
display(top20_knn)

In [ ]:
# 43. VISUALIZAÇÕES DO ÍNDICE DE RISCO (KNN)

import matplotlib.pyplot as plt

# 43.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco_knn["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Thresholds corretos
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo/Médio (30%)")
ax.axvline(0.50, color="red", linestyle="--", linewidth=1.5, label="Médio/Alto (50%)")
ax.axvline(0.70, color="darkred", linestyle="--", linewidth=1.5, label="Alto/Crítico (70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída (KNN)")

ax.legend()

plt.tight_layout()
plt.savefig("distribuicao_probabilidades_knn.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 43.2 Contagem por categoria (KNN)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",
    "Médio": "#f39c12",
    "Alto": "#e74c3c",
    "Crítico": "#8e0000"
}

fig, ax = plt.subplots(figsize=(9, 5))

# Valores das categorias
vals = [contagem_knn.get(c, 0) for c in ORDEM]

bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

# Labels (nº + %)
for bar, cat in zip(bars, ORDEM):
    n   = contagem_knn.get(cat, 0)
    pct = percentagem_knn.get(cat, 0.0)
    
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco (KNN)")

plt.tight_layout()
plt.savefig("categorias_risco_knn.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 43.3 Boxplot por categoria (KNN)

import matplotlib.pyplot as plt
import seaborn as sns

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = ["#2ecc71", "#f39c12", "#e74c3c", "#8e0000"]

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df_risco_knn,
    x="nivel_risco",
    y="prob_saida",
    order=ORDEM,
    palette=cores,
    ax=ax
)

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saída")
ax.set_title("Distribuição das Probabilidades por Categoria (KNN)")

plt.tight_layout()
plt.savefig("boxplot_risco_knn.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 44. RESUMO FINAL (KNN)

print("=" * 55)
print("RESUMO — KNN")
print("=" * 55)

print(f"  Modelo:         KNN (parâmetros default)")
print(f"  Colaboradores:  {len(df_risco_knn)}")

print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12}  {resultados_treino_knn[metrica]:>8.4f}  {resultados_teste_knn[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")

print(f"  Baixo:    prob < 30%        -> {contagem_knn.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem_knn.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem_knn.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem_knn.get('Crítico', 0)} colaboradores")

print("=" * 55)

## 4. Candidato - Extra Trees 


In [ ]:
# 45. TREINO — EXTRA TREES
pipeline_et = Pipeline([("et", ExtraTreesClassifier(random_state=42)) ])

pipeline_et.fit(X_train, y_train)
print("Modelo Extra Trees treinado.")

In [ ]:
# 46. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE (EXTRA TREES)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }

In [ ]:
# 46.1 TREINO
resultados_treino_et = avaliar_modelo(pipeline_et, X_train, y_train, "Treino")

In [ ]:
# 46.2 TESTE
resultados_teste_et  = avaliar_modelo(pipeline_et, X_test,  y_test,  "Teste")

In [ ]:
# 47. COMPARAÇÃO TREINO vs TESTE (EXTRA TREES)

print("===== COMPARAÇÃO TREINO vs TESTE (EXTRA TREES) =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*44}")

# Iterar sobre as métricas guardadas no dicionário
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino_et[metrica]
    val_teste  = resultados_teste_et[metrica]
    diff       = val_treino - val_teste
    
    # Formatação do nome da métrica
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

# Validação automática de Overfitting focada no F1-Score
diff_f1_et = resultados_treino_et["f1"] - resultados_teste_et["f1"]

print() # Linha em branco para limpeza visual
if diff_f1_et > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting.")

In [ ]:
# 48. CURVAS ROC SOBREPOSTAS (EXTRA TREES)
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Treino - Usando os resultados do Extra Trees
RocCurveDisplay.from_predictions(
    resultados_treino_et["y"], resultados_treino_et["y_proba"],
    name=f"Treino (AUC={resultados_treino_et['auc']:.3f})", ax=ax, color="steelblue"
)

# Curva de Teste - Usando os resultados do Extra Trees
RocCurveDisplay.from_predictions(
    resultados_teste_et["y"], resultados_teste_et["y_proba"],
    name=f"Teste  (AUC={resultados_teste_et['auc']:.3f})", ax=ax, color="darkorange"
)

# Linha de referência (o "acaso")
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_title("Curva ROC — Extra Trees (Treino vs Teste)")
plt.tight_layout()
# Salvando com um nome de arquivo diferente
plt.savefig("roc_treino_vs_teste_et.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 49. GERAR PROBABILIDADES DE SAÍDA (dataset completo - EXTRA TREES)
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

# Evitar erros caso a Attrition_bin não esteja no CSV
X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"], errors='ignore')
X_completo = X_completo.select_dtypes(include=[np.number])

# Usamos df_risco_et para não sobrescrever o teu trabalho anterior
df_risco_et = df_completo.copy()

# A MUDANÇA PRINCIPAL: Chamar o pipeline do Extra Trees
df_risco_et["prob_saida"] = pipeline_et.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco_et)} colaboradores (Extra Trees).")
print(df_risco_et["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 50. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO (EXTRA TREES)

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

# Aplicar a função ao DataFrame do Extra Trees
df_risco_et["nivel_risco"] = df_risco_et["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem_et    = df_risco_et["nivel_risco"].value_counts()
percentagem_et = df_risco_et["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO (EXTRA TREES) =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n   = contagem_et.get(cat, 0)
    pct = percentagem_et.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
# 51. ANÁLISE POR CATEGORIA DE RISCO (EXTRA TREES)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cols_analise = ["prob_saida", "Attrition_bin"]

# Adicionar variáveis de negócio para caracterizar cada nível de risco
for col in [
    "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
    "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"
]:
    if col in df_risco_et.columns:
        cols_analise.append(col)

print("\n===== PERFIL MÉDIO POR CATEGORIA DE RISCO (EXTRA TREES) =====")

perfil_et = (
    df_risco_et
    .groupby("nivel_risco")[cols_analise]
    .mean()
    .reindex(ORDEM)
    .round(3)
)

display(perfil_et)

In [ ]:
# 52. TOP 20 COLABORADORES COM MAIOR RISCO (EXTRA TREES)

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco_et.columns:
        cols_top.append(col)

top20_et = df_risco_et.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA (EXTRA TREES) =====")
display(top20_et)

In [ ]:
# 53. VISUALIZAÇÕES DO ÍNDICE DE RISCO (EXTRA TREES)

import matplotlib.pyplot as plt

# 53.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco_et["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Thresholds corretos
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo/Médio (30%)")
ax.axvline(0.50, color="red", linestyle="--", linewidth=1.5, label="Médio/Alto (50%)")
ax.axvline(0.70, color="darkred", linestyle="--", linewidth=1.5, label="Alto/Crítico (70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída (Extra Trees)")

ax.legend()

plt.tight_layout()
plt.savefig("distribuicao_probabilidades_et.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 53.2 Contagem por categoria (EXTRA TREES)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",
    "Médio": "#f39c12",
    "Alto": "#e74c3c",
    "Crítico": "#8e0000"
}

fig, ax = plt.subplots(figsize=(9, 5))

# Valores das categorias
vals = [contagem_et.get(c, 0) for c in ORDEM]

bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

# Labels (nº + %)
for bar, cat in zip(bars, ORDEM):
    n   = contagem_et.get(cat, 0)
    pct = percentagem_et.get(cat, 0.0)
    
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco (Extra Trees)")

plt.tight_layout()
plt.savefig("categorias_risco_et.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 53.3 Boxplot por categoria (EXTRA TREES)

import matplotlib.pyplot as plt
import seaborn as sns

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = ["#2ecc71", "#f39c12", "#e74c3c", "#8e0000"]

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df_risco_et,
    x="nivel_risco",
    y="prob_saida",
    order=ORDEM,
    palette=cores,
    ax=ax
)

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saída")
ax.set_title("Distribuição das Probabilidades por Categoria (Extra Trees)")

plt.tight_layout()
plt.savefig("boxplot_risco_et.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 54. RESUMO FINAL (EXTRA TREES)

print("=" * 55)
print("RESUMO — EXTRA TREES")
print("=" * 55)

print(f"  Modelo:         Extra Trees (parâmetros default)")
print(f"  Colaboradores:  {len(df_risco_et)}")

print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12}  {resultados_treino_et[metrica]:>8.4f}  {resultados_teste_et[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")

print(f"  Baixo:    prob < 30%        -> {contagem_et.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem_et.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem_et.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem_et.get('Crítico', 0)} colaboradores")

print("=" * 55)

## 5. Candidato  - SVM (SUPPORT VECTOR MACHINE)
Modelo SVM com StandardScaler e classificação de risco.

In [ ]:
# 55. TREINO — SUPPORT VECTOR MACHINE (SVM)
pipeline_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(probability=True, random_state=42)) 
])

pipeline_svm.fit(X_train, y_train)
print("Modelo Support Vector Machine (SVM) treinado.")

In [ ]:
# 56. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE (SVM)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }

In [ ]:
# 56.1 TREINO
resultados_treino_svm = avaliar_modelo(pipeline_svm, X_train, y_train, "Treino")

In [ ]:
# 56.2 TESTE
resultados_teste_svm  = avaliar_modelo(pipeline_svm, X_test,  y_test,  "Teste")

In [ ]:
# 57. COMPARAÇÃO TREINO vs TESTE (SVM)

print("===== COMPARAÇÃO TREINO vs TESTE (SVM) =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*44}")

# Iterar sobre as métricas guardadas no dicionário do SVM
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino_svm[metrica]
    val_teste  = resultados_teste_svm[metrica]
    diff       = val_treino - val_teste
    
    # Formatação do nome da métrica
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

# Validação automática de Overfitting focada no F1-Score
diff_f1_svm = resultados_treino_svm["f1"] - resultados_teste_svm["f1"]

print() # Linha em branco para limpeza visual
if diff_f1_svm > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting (Modelo Estável).")

In [ ]:
# 58. CURVAS ROC SOBREPOSTAS (SVM)
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Treino - Usando os resultados do SVM
RocCurveDisplay.from_predictions(
    resultados_treino_svm["y"], resultados_treino_svm["y_proba"],
    name=f"Treino (AUC={resultados_treino_svm['auc']:.3f})", ax=ax, color="steelblue"
)

# Curva de Teste - Usando os resultados do SVM
RocCurveDisplay.from_predictions(
    resultados_teste_svm["y"], resultados_teste_svm["y_proba"],
    name=f"Teste  (AUC={resultados_teste_svm['auc']:.3f})", ax=ax, color="darkorange"
)

# Linha de referência (o "acaso")
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_title("Curva ROC — SVM (Treino vs Teste)")
plt.tight_layout()

# Salvando com o nome do modelo atual
plt.savefig("roc_treino_vs_teste_svm.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 59. GERAR PROBABILIDADES DE SAÍDA (dataset completo - SVM)
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

# Evitar erros caso a Attrition_bin não esteja no CSV
X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"], errors='ignore')
X_completo = X_completo.select_dtypes(include=[np.number])

# Usamos df_risco_svm para não sobrescrever o trabalho do Extra Trees ou KNN
df_risco_svm = df_completo.copy()

# A MUDANÇA PRINCIPAL: Chamar o pipeline do SVM
df_risco_svm["prob_saida"] = pipeline_svm.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco_svm)} colaboradores (SVM).")
print(df_risco_svm["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 60. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO (SVM)

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

# Aplicar a função ao DataFrame do SVM
df_risco_svm["nivel_risco"] = df_risco_svm["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem_svm    = df_risco_svm["nivel_risco"].value_counts()
percentagem_svm = df_risco_svm["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO (SVM) =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n   = contagem_svm.get(cat, 0)
    pct = percentagem_svm.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
# 61. ANÁLISE POR CATEGORIA DE RISCO (SVM)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cols_analise = ["prob_saida", "Attrition_bin"]

# Adicionar variáveis de negócio para caracterizar cada nível de risco
for col in [
    "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
    "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"
]:
    if col in df_risco_svm.columns:
        cols_analise.append(col)

print("\n===== PERFIL MÉDIO POR CATEGORIA DE RISCO (SVM) =====")

perfil_svm = (
    df_risco_svm
    .groupby("nivel_risco")[cols_analise]
    .mean()
    .reindex(ORDEM)
    .round(3)
)

display(perfil_svm)

In [ ]:
# 62. TOP 20 COLABORADORES COM MAIOR RISCO (SVM)

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco_svm.columns:
        cols_top.append(col)

# Usar o DataFrame do SVM para encontrar os 20 casos mais críticos
top20_svm = df_risco_svm.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA (SVM) =====")
display(top20_svm)

In [ ]:
# 63. VISUALIZAÇÕES DO ÍNDICE DE RISCO (SVM)

import matplotlib.pyplot as plt

# 63.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco_svm["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Thresholds corretos
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo/Médio (30%)")
ax.axvline(0.50, color="red", linestyle="--", linewidth=1.5, label="Médio/Alto (50%)")
ax.axvline(0.70, color="darkred", linestyle="--", linewidth=1.5, label="Alto/Crítico (70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída (SVM)")

ax.legend()

plt.tight_layout()
plt.savefig("distribuicao_probabilidades_svm.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 63.2 Contagem por categoria (SVM)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",
    "Médio": "#f39c12",
    "Alto": "#e74c3c",
    "Crítico": "#8e0000"
}

fig, ax = plt.subplots(figsize=(9, 5))

# Valores das categorias
vals = [contagem_svm.get(c, 0) for c in ORDEM]

bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

# Labels (nº + %)
for bar, cat in zip(bars, ORDEM):
    n   = contagem_svm.get(cat, 0)
    pct = percentagem_svm.get(cat, 0.0)
    
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco (SVM)")

plt.tight_layout()
plt.savefig("categorias_risco_svm.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 63.3 Boxplot por categoria (SVM)

import matplotlib.pyplot as plt
import seaborn as sns

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = ["#2ecc71", "#f39c12", "#e74c3c", "#8e0000"]

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df_risco_svm,
    x="nivel_risco",
    y="prob_saida",
    order=ORDEM,
    palette=cores,
    ax=ax
)

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saída")
ax.set_title("Distribuição das Probabilidades por Categoria (SVM)")

plt.tight_layout()
plt.savefig("boxplot_risco_svm.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 64. RESUMO FINAL (SVM)

print("=" * 55)
print("RESUMO — SVM")
print("=" * 55)

print(f"  Modelo:         Support Vector Machine (SVC)")
print(f"  Colaboradores:  {len(df_risco_svm)}")

print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12}  {resultados_treino_svm[metrica]:>8.4f}  {resultados_teste_svm[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")

print(f"  Baixo:    prob < 30%        -> {contagem_svm.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem_svm.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem_svm.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem_svm.get('Crítico', 0)} colaboradores")

print("=" * 55)

## 6. Candidato  - ADABOOST 

In [ ]:
# 65. TREINO — ADABOOST
pipeline_ada = Pipeline([("ada", AdaBoostClassifier(random_state=42, n_estimators=100)) ])

pipeline_ada.fit(X_train, y_train)
print("Modelo AdaBoost treinado.")

In [ ]:
# 66. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE (ADABOOST)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }

In [ ]:
# 66.1 TREINO
resultados_treino_ada = avaliar_modelo(pipeline_ada, X_train, y_train, "Treino")

In [ ]:
# 66.2 TESTE 
resultados_teste_ada  = avaliar_modelo(pipeline_ada, X_test,  y_test,  "Teste")

In [ ]:
# 67. COMPARAÇÃO TREINO vs TESTE (ADABOOST)

print("===== COMPARAÇÃO TREINO vs TESTE (ADABOOST) =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*44}")

# Iterar sobre as métricas guardadas no dicionário do AdaBoost
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino_ada[metrica]
    val_teste  = resultados_teste_ada[metrica]
    diff       = val_treino - val_teste
    
    # Formatação do nome da métrica
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

# Validação automática de Overfitting focada no F1-Score
diff_f1_ada = resultados_treino_ada["f1"] - resultados_teste_ada["f1"]

print() # Linha em branco para limpeza visual
if diff_f1_ada > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting (Modelo Estável).")

In [ ]:
# 68. CURVAS ROC SOBREPOSTAS (ADABOOST)
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Treino - Usando os resultados do AdaBoost
RocCurveDisplay.from_predictions(
    resultados_treino_ada["y"], resultados_treino_ada["y_proba"],
    name=f"Treino (AUC={resultados_treino_ada['auc']:.3f})", ax=ax, color="steelblue"
)

# Curva de Teste - Usando os resultados do AdaBoost
RocCurveDisplay.from_predictions(
    resultados_teste_ada["y"], resultados_teste_ada["y_proba"],
    name=f"Teste  (AUC={resultados_teste_ada['auc']:.3f})", ax=ax, color="darkorange"
)

# Linha de referência (o "acaso")
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_title("Curva ROC — AdaBoost (Treino vs Teste)")
plt.tight_layout()

# Salvando com o nome do modelo atual
plt.savefig("roc_treino_vs_teste_ada.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 69. GERAR PROBABILIDADES DE SAÍDA (dataset completo - ADABOOST)
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

# Evitar erros caso a Attrition_bin não esteja no CSV
X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"], errors='ignore')
X_completo = X_completo.select_dtypes(include=[np.number])

# Usamos df_risco_ada para não sobrescrever o trabalho do SVM, Extra Trees ou KNN
df_risco_ada = df_completo.copy()

# A MUDANÇA PRINCIPAL: Chamar o pipeline do AdaBoost
df_risco_ada["prob_saida"] = pipeline_ada.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco_ada)} colaboradores (AdaBoost).")
print(df_risco_ada["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 70. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO (ADABOOST)

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

# Aplicar a função ao DataFrame do AdaBoost
df_risco_ada["nivel_risco"] = df_risco_ada["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem_ada    = df_risco_ada["nivel_risco"].value_counts()
percentagem_ada = df_risco_ada["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO (ADABOOST) =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n   = contagem_ada.get(cat, 0)
    pct = percentagem_ada.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
# 71. ANÁLISE POR CATEGORIA DE RISCO (ADABOOST)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cols_analise = ["prob_saida", "Attrition_bin"]

# Adicionar variáveis de negócio para caracterizar cada nível de risco
for col in [
    "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
    "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"
]:
    if col in df_risco_ada.columns:
        cols_analise.append(col)

print("\n===== PERFIL MÉDIO POR CATEGORIA DE RISCO (ADABOOST) =====")

perfil_ada = (
    df_risco_ada
    .groupby("nivel_risco")[cols_analise]
    .mean()
    .reindex(ORDEM)
    .round(3)
)

display(perfil_ada)

In [ ]:
# 72. TOP 20 COLABORADORES COM MAIOR RISCO (ADABOOST)

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco_ada.columns:
        cols_top.append(col)

# Usar o DataFrame do AdaBoost para encontrar os 20 casos mais críticos
top20_ada = df_risco_ada.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA (ADABOOST) =====")
display(top20_ada)

In [ ]:
# 73. VISUALIZAÇÕES DO ÍNDICE DE RISCO (ADABOOST)

import matplotlib.pyplot as plt

# 73.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco_ada["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Thresholds corretos
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo/Médio (30%)")
ax.axvline(0.50, color="red", linestyle="--", linewidth=1.5, label="Médio/Alto (50%)")
ax.axvline(0.70, color="darkred", linestyle="--", linewidth=1.5, label="Alto/Crítico (70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída (AdaBoost)")

ax.legend()

plt.tight_layout()
plt.savefig("distribuicao_probabilidades_ada.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 73.2 Contagem por categoria (ADABOOST)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",
    "Médio": "#f39c12",
    "Alto": "#e74c3c",
    "Crítico": "#8e0000"
}

fig, ax = plt.subplots(figsize=(9, 5))

# Valores das categorias
vals = [contagem_ada.get(c, 0) for c in ORDEM]

bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

# Labels (nº + %)
for bar, cat in zip(bars, ORDEM):
    n   = contagem_ada.get(cat, 0)
    pct = percentagem_ada.get(cat, 0.0)
    
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco (AdaBoost)")

plt.tight_layout()
plt.savefig("categorias_risco_ada.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 73.3 Boxplot por categoria (ADABOOST)

import matplotlib.pyplot as plt
import seaborn as sns

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = ["#2ecc71", "#f39c12", "#e74c3c", "#8e0000"]

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df_risco_ada,
    x="nivel_risco",
    y="prob_saida",
    order=ORDEM,
    palette=cores,
    ax=ax
)

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saída")
ax.set_title("Distribuição das Probabilidades por Categoria (AdaBoost)")

plt.tight_layout()
plt.savefig("boxplot_risco_ada.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 74. RESUMO FINAL (ADABOOST)

print("=" * 55)
print("RESUMO — ADABOOST")
print("=" * 55)

print(f"  Modelo:         AdaBoost (100 estimadores)")
print(f"  Colaboradores:  {len(df_risco_ada)}")

print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12}  {resultados_treino_ada[metrica]:>8.4f}  {resultados_teste_ada[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")

print(f"  Baixo:    prob < 30%        -> {contagem_ada.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem_ada.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem_ada.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem_ada.get('Crítico', 0)} colaboradores")

print("=" * 55)

## 7. Candidato  - LIGHTGBM (Light Gradient Boosting Machine)

In [ ]:
# 75. TREINO — LIGHTGBM
pipeline_lgb = Pipeline([ ("lgb", LGBMClassifier(random_state=42, verbose=-1)) ])

pipeline_lgb.fit(X_train, y_train)
print("Modelo LightGBM treinado com sucesso!")

In [ ]:
# 76. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE (LIGHTGBM)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }

In [ ]:
# 76.1 TREINO
resultados_treino_lgb = avaliar_modelo(pipeline_lgb, X_train, y_train, "Treino")

In [ ]:
# 76.2 TESTE
resultados_teste_lgb  = avaliar_modelo(pipeline_lgb, X_test,  y_test,  "Teste")

In [ ]:
# 77. COMPARAÇÃO TREINO vs TESTE (LIGHTGBM)

print("===== COMPARAÇÃO TREINO vs TESTE (LIGHTGBM) =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*44}")

# Iterar sobre as métricas guardadas no dicionário do LightGBM
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino_lgb[metrica]
    val_teste  = resultados_teste_lgb[metrica]
    diff       = val_treino - val_teste
    
    # Formatação do nome da métrica
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

# Validação automática de Overfitting focada no F1-Score
diff_f1_lgb = resultados_treino_lgb["f1"] - resultados_teste_lgb["f1"]

print() # Linha em branco para limpeza visual
if diff_f1_lgb > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting (Modelo Estável).")

In [ ]:
# 78. CURVAS ROC SOBREPOSTAS (LIGHTGBM)
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Treino - Usando os resultados do LightGBM
RocCurveDisplay.from_predictions(
    resultados_treino_lgb["y"], resultados_treino_lgb["y_proba"],
    name=f"Treino (AUC={resultados_treino_lgb['auc']:.3f})", ax=ax, color="steelblue"
)

# Curva de Teste - Usando os resultados do LightGBM
RocCurveDisplay.from_predictions(
    resultados_teste_lgb["y"], resultados_teste_lgb["y_proba"],
    name=f"Teste  (AUC={resultados_teste_lgb['auc']:.3f})", ax=ax, color="darkorange"
)

# Linha de referência (o "acaso")
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_title("Curva ROC — LightGBM (Treino vs Teste)")
plt.tight_layout()

# Salvando com o nome do modelo atual
plt.savefig("roc_treino_vs_teste_lgb.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 79. GERAR PROBABILIDADES DE SAÍDA (dataset completo - LIGHTGBM)
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

# Evitar erros caso a Attrition_bin não esteja no CSV
X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"], errors='ignore')
X_completo = X_completo.select_dtypes(include=[np.number])

# Usamos df_risco_lgb para não sobrescrever o trabalho do AdaBoost, SVM ou outros
df_risco_lgb = df_completo.copy()

# A MUDANÇA PRINCIPAL: Chamar o pipeline do LightGBM
df_risco_lgb["prob_saida"] = pipeline_lgb.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco_lgb)} colaboradores (LightGBM).")
print(df_risco_lgb["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 80. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO (LIGHTGBM)

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

# Aplicar a função ao DataFrame do LightGBM
df_risco_lgb["nivel_risco"] = df_risco_lgb["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem_lgb    = df_risco_lgb["nivel_risco"].value_counts()
percentagem_lgb = df_risco_lgb["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO (LIGHTGBM) =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n   = contagem_lgb.get(cat, 0)
    pct = percentagem_lgb.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
# 81. ANÁLISE POR CATEGORIA DE RISCO (LIGHTGBM)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cols_analise = ["prob_saida", "Attrition_bin"]

# Adicionar variáveis de negócio para caracterizar cada nível de risco
for col in [
    "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
    "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"
]:
    if col in df_risco_lgb.columns:
        cols_analise.append(col)

print("\n===== PERFIL MÉDIO POR CATEGORIA DE RISCO (LIGHTGBM) =====")

perfil_lgb = (
    df_risco_lgb
    .groupby("nivel_risco")[cols_analise]
    .mean()
    .reindex(ORDEM)
    .round(3)
)

display(perfil_lgb)

In [ ]:
# 82. TOP 20 COLABORADORES COM MAIOR RISCO (LIGHTGBM)

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco_lgb.columns:
        cols_top.append(col)

# Usar o DataFrame do LightGBM para encontrar os 20 casos mais críticos
top20_lgb = df_risco_lgb.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA (LIGHTGBM) =====")
display(top20_lgb)

In [ ]:
# 83. VISUALIZAÇÕES DO ÍNDICE DE RISCO (LIGHTGBM)

import matplotlib.pyplot as plt

# 83.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco_lgb["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Thresholds corretos
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo/Médio (30%)")
ax.axvline(0.50, color="red", linestyle="--", linewidth=1.5, label="Médio/Alto (50%)")
ax.axvline(0.70, color="darkred", linestyle="--", linewidth=1.5, label="Alto/Crítico (70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída (LightGBM)")

ax.legend()

plt.tight_layout()
plt.savefig("distribuicao_probabilidades_lgb.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 83.2 Contagem por categoria (LIGHTGBM)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",
    "Médio": "#f39c12",
    "Alto": "#e74c3c",
    "Crítico": "#8e0000"
}

fig, ax = plt.subplots(figsize=(9, 5)) 

vals = [contagem_lgb.get(c, 0) for c in ORDEM]

bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

for bar, cat in zip(bars, ORDEM):
    n   = contagem_lgb.get(cat, 0)
    pct = percentagem_lgb.get(cat, 0.0)
    
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco (LightGBM)")

plt.tight_layout()
plt.savefig("categorias_risco_lgb.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 83.3 Boxplot por categoria (LIGHTGBM)

import matplotlib.pyplot as plt
import seaborn as sns

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = ["#2ecc71", "#f39c12", "#e74c3c", "#8e0000"]

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df_risco_lgb,
    x="nivel_risco",
    y="prob_saida",
    order=ORDEM,
    palette=cores,
    ax=ax
)

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saída")
ax.set_title("Distribuição das Probabilidades por Categoria (LightGBM)")

plt.tight_layout()
plt.savefig("boxplot_risco_lgb.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 84. RESUMO FINAL (LIGHTGBM)

print("=" * 55)
print("RESUMO — LIGHTGBM")
print("=" * 55)

print(f"  Modelo:         LightGBM (100 estimadores)")
print(f"  Colaboradores:  {len(df_risco_lgb)}")

print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12}  {resultados_treino_lgb[metrica]:>8.4f}  {resultados_teste_lgb[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")

print(f"  Baixo:    prob < 30%        -> {contagem_lgb.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem_lgb.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem_lgb.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem_lgb.get('Crítico', 0)} colaboradores")

print("=" * 55)

## 8. Candidato  - XGBOOST

In [ ]:
# 85.TREINO — XGBOOST
pipeline_xgb = Pipeline([("xgb", XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)) ])

pipeline_xgb.fit(X_train, y_train)
print("Modelo XGBoost treinado com sucesso!")

In [ ]:
# 86. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE (XGBOOST)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }

In [ ]:
# 86.1 TREINO
resultados_treino_xgb = avaliar_modelo(pipeline_xgb, X_train, y_train, "Treino")

In [ ]:
# 86.2 TESTE
resultados_teste_xgb  = avaliar_modelo(pipeline_xgb, X_test,  y_test,  "Teste")

In [ ]:
# 87. COMPARAÇÃO TREINO vs TESTE (XGBOOST)

print("===== COMPARAÇÃO TREINO vs TESTE (XGBOOST) =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*44}")

# Iterar sobre as métricas guardadas no dicionário do XGBoost
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino_xgb[metrica]
    val_teste  = resultados_teste_xgb[metrica]
    diff       = val_treino - val_teste
    
    # Formatação do nome da métrica
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

# Validação automática de Overfitting focada no F1-Score
diff_f1_xgb = resultados_treino_xgb["f1"] - resultados_teste_xgb["f1"]

print() # Linha em branco para limpeza visual
if diff_f1_xgb > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting (Modelo Estável).")

In [ ]:
# 88. CURVAS ROC SOBREPOSTAS (XGBOOST)
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Treino - Usando os resultados do XGBoost
RocCurveDisplay.from_predictions(
    resultados_treino_xgb["y"], resultados_treino_xgb["y_proba"],
    name=f"Treino (AUC={resultados_treino_xgb['auc']:.3f})", ax=ax, color="steelblue"
)

# Curva de Teste - Usando os resultados do XGBoost
RocCurveDisplay.from_predictions(
    resultados_teste_xgb["y"], resultados_teste_xgb["y_proba"],
    name=f"Teste  (AUC={resultados_teste_xgb['auc']:.3f})", ax=ax, color="darkorange"
)

# Linha de referência (o "acaso")
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_title("Curva ROC — XGBoost (Treino vs Teste)")
plt.tight_layout()

# Salvando com o nome do modelo atual
plt.savefig("roc_treino_vs_teste_xgb.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 89. GERAR PROBABILIDADES DE SAÍDA (dataset completo - XGBOOST)
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

# Evitar erros caso a Attrition_bin não esteja no CSV
X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"], errors='ignore')
X_completo = X_completo.select_dtypes(include=[np.number])

# Usamos df_risco_xgb para não sobrescrever o trabalho do LightGBM, AdaBoost ou outros
df_risco_xgb = df_completo.copy()

# A MUDANÇA PRINCIPAL: Chamar o pipeline do XGBoost
df_risco_xgb["prob_saida"] = pipeline_xgb.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco_xgb)} colaboradores (XGBoost).")
print(df_risco_xgb["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 90. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO (XGBOOST)

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

# Aplicar a função ao DataFrame do XGBoost
df_risco_xgb["nivel_risco"] = df_risco_xgb["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem_xgb    = df_risco_xgb["nivel_risco"].value_counts()
percentagem_xgb = df_risco_xgb["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO (XGBOOST) =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n   = contagem_xgb.get(cat, 0)
    pct = percentagem_xgb.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
# 91. ANÁLISE POR CATEGORIA DE RISCO (XGBOOST)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cols_analise = ["prob_saida", "Attrition_bin"]

# Adicionar variáveis de negócio para caracterizar cada nível de risco
for col in [
    "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
    "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"
]:
    if col in df_risco_xgb.columns:
        cols_analise.append(col)

print("\n===== PERFIL MÉDIO POR CATEGORIA DE RISCO (XGBOOST) =====")

perfil_xgb = (
    df_risco_xgb
    .groupby("nivel_risco")[cols_analise]
    .mean()
    .reindex(ORDEM)
    .round(3)
)

display(perfil_xgb)

In [ ]:
# 92. TOP 20 COLABORADORES COM MAIOR RISCO (XGBOOST)

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco_xgb.columns:
        cols_top.append(col)

# Usar o DataFrame do XGBoost para encontrar os 20 casos mais críticos
top20_xgb = df_risco_xgb.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA (XGBOOST) =====")
display(top20_xgb)

In [ ]:
# 93. VISUALIZAÇÕES DO ÍNDICE DE RISCO (XGBOOST)

import matplotlib.pyplot as plt

# 93.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco_xgb["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Thresholds corretos
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo/Médio (30%)")
ax.axvline(0.50, color="red", linestyle="--", linewidth=1.5, label="Médio/Alto (50%)")
ax.axvline(0.70, color="darkred", linestyle="--", linewidth=1.5, label="Alto/Crítico (70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída (XGBoost)")

ax.legend()

plt.tight_layout()
plt.savefig("distribuicao_probabilidades_xgb.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 93.2 Contagem por categoria (XGBOOST)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",
    "Médio": "#f39c12",
    "Alto": "#e74c3c",
    "Crítico": "#8e0000"
}

fig, ax = plt.subplots(figsize=(9, 5)) 

# Valores das categorias
vals = [contagem_xgb.get(c, 0) for c in ORDEM]

bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

# Labels (nº + %)
for bar, cat in zip(bars, ORDEM):
    n   = contagem_xgb.get(cat, 0)
    pct = percentagem_xgb.get(cat, 0.0)
    
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco (XGBoost)")

plt.tight_layout()
plt.savefig("categorias_risco_xgb.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 93.3 Boxplot por categoria (XGBOOST)

import matplotlib.pyplot as plt
import seaborn as sns

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = ["#2ecc71", "#f39c12", "#e74c3c", "#8e0000"]

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df_risco_xgb,
    x="nivel_risco",
    y="prob_saida",
    order=ORDEM,
    palette=cores,
    ax=ax
)

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saída")
ax.set_title("Distribuição das Probabilidades por Categoria (XGBoost)")

plt.tight_layout()
plt.savefig("boxplot_risco_xgb.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 94. RESUMO FINAL (XGBOOST)

print("=" * 55)
print("RESUMO — XGBOOST")
print("=" * 55)

print(f"  Modelo:         XGBoost (100 estimadores)")
print(f"  Colaboradores:  {len(df_risco_xgb)}")

print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12}  {resultados_treino_xgb[metrica]:>8.4f}  {resultados_teste_xgb[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")

print(f"  Baixo:    prob < 30%        -> {contagem_xgb.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem_xgb.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem_xgb.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem_xgb.get('Crítico', 0)} colaboradores")

print("=" * 55)

## 9. Candidato  - RANDOM FOREST

In [ ]:
# 95. TREINO — RANDOM FOREST

pipeline_rf = Pipeline([("rf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)) ])

pipeline_rf.fit(X_train, y_train)
print("Modelo Random Forest treinado com sucesso!")

In [ ]:
# 96. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE (RANDOM FOREST)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# Nota: A função 'avaliar_modelo' já foi definida anteriormente, 
# mas mantemo-la aqui para garantir que o código é autoexecutável.

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"===== METRICAS — {nome_conjunto} =====")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print()
    print(classification_report(y, y_pred, target_names=["Permaneceu", "Saiu"]))

    return {
        "conjunto":  nome_conjunto,
        "acc":       acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       auc,
        "y":         y,
        "y_proba":   y_pred_proba
    }

In [ ]:
# 96.1 TREINO
resultados_treino_rf = avaliar_modelo(pipeline_rf, X_train, y_train, "Treino")

In [ ]:
# 96.2 TESTE
resultados_teste_rf  = avaliar_modelo(pipeline_rf, X_test,  y_test,  "Teste")

In [ ]:
# 97. COMPARAÇÃO TREINO vs TESTE (RANDOM FOREST)

print("===== COMPARAÇÃO TREINO vs TESTE (RANDOM FOREST) =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*44}")

# Iterar sobre as métricas guardadas no dicionário do Random Forest
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino_rf[metrica]
    val_teste  = resultados_teste_rf[metrica]
    diff       = val_treino - val_teste
    
    # Formatação do nome da métrica
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

# Validação automática de Overfitting focada no F1-Score
diff_f1_rf = resultados_treino_rf["f1"] - resultados_teste_rf["f1"]

print() # Linha em branco para limpeza visual
if diff_f1_rf > 0.10:
    print("  --> Sinal de OVERFITTING: o modelo decorou o treino mas generaliza mal.")
else:
    print("  --> Sem sinais evidentes de overfitting (Modelo Estável).")

In [ ]:
# 98. CURVAS ROC SOBREPOSTAS (XGBOOST)
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))

# Curva de Treino - Usando os resultados do XGBoost
RocCurveDisplay.from_predictions(
    resultados_treino_xgb["y"], resultados_treino_xgb["y_proba"],
    name=f"Treino (AUC={resultados_treino_xgb['auc']:.3f})", ax=ax, color="steelblue"
)

# Curva de Teste - Usando os resultados do XGBoost
RocCurveDisplay.from_predictions(
    resultados_teste_xgb["y"], resultados_teste_xgb["y_proba"],
    name=f"Teste  (AUC={resultados_teste_xgb['auc']:.3f})", ax=ax, color="darkorange"
)

# Linha de referência (o "acaso")
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_title("Curva ROC — XGBoost (Treino vs Teste)")
plt.tight_layout()

# Salvando com o nome do modelo atual
plt.savefig("roc_treino_vs_teste_xgb.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 99. GERAR PROBABILIDADES DE SAÍDA (dataset completo - RANDOM FOREST)
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
df_completo = pd.read_csv(url)

# Aplicar a mesma preparação de features
cols_remover = ["Attrition", "OverTime", "Gender", "BusinessTravel", 
                "Department", "EducationField", "JobRole", "MaritalStatus"]
cols_remover = [c for c in cols_remover if c in df_completo.columns]

# Evitar erros caso a Attrition_bin não esteja no CSV
X_completo = df_completo.drop(columns=cols_remover + ["Attrition_bin"], errors='ignore')
X_completo = X_completo.select_dtypes(include=[np.number])

# Usamos df_risco_rf para não sobrescrever o trabalho do XGBoost, LightGBM ou outros
df_risco_rf = df_completo.copy()

# A MUDANÇA PRINCIPAL: Chamar o pipeline do Random Forest
df_risco_rf["prob_saida"] = pipeline_rf.predict_proba(X_completo)[:, 1]

print(f"Probabilidades geradas para {len(df_risco_rf)} colaboradores (Random Forest).")
print(df_risco_rf["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 100. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO (RANDOM FOREST)

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

# Aplicar a função ao DataFrame do Random Forest
df_risco_rf["nivel_risco"] = df_risco_rf["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem_rf    = df_risco_rf["nivel_risco"].value_counts()
percentagem_rf = df_risco_rf["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO (RANDOM FOREST) =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n   = contagem_rf.get(cat, 0)
    pct = percentagem_rf.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
# 101. ANÁLISE POR CATEGORIA DE RISCO (RANDOM FOREST)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cols_analise = ["prob_saida", "Attrition_bin"]

# Adicionar variáveis de negócio para caracterizar cada nível de risco
for col in [
    "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
    "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"
]:
    if col in df_risco_rf.columns:
        cols_analise.append(col)

print("\n===== PERFIL MÉDIO POR CATEGORIA DE RISCO (RANDOM FOREST) =====")

perfil_rf = (
    df_risco_rf
    .groupby("nivel_risco")[cols_analise]
    .mean()
    .reindex(ORDEM)
    .round(3)
)

display(perfil_rf)

In [ ]:
# 102. TOP 20 COLABORADORES COM MAIOR RISCO (RANDOM FOREST)

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco_rf.columns:
        cols_top.append(col)

# Usar o DataFrame do Random Forest para encontrar os 20 casos mais críticos
top20_rf = df_risco_rf.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA (RANDOM FOREST) =====")
display(top20_rf)

In [ ]:
# 103. VISUALIZAÇÕES DO ÍNDICE DE RISCO (RANDOM FOREST)

import matplotlib.pyplot as plt

# 103.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_risco_rf["prob_saida"], bins=40, color="steelblue", edgecolor="white")

# Thresholds corretos
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Baixo/Médio (30%)")
ax.axvline(0.50, color="red", linestyle="--", linewidth=1.5, label="Médio/Alto (50%)")
ax.axvline(0.70, color="darkred", linestyle="--", linewidth=1.5, label="Alto/Crítico (70%)")

ax.set_xlabel("Probabilidade de Saída")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Distribuição das Probabilidades de Saída (Random Forest)")

ax.legend()

plt.tight_layout()
plt.savefig("distribuicao_probabilidades_rf.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 103.2 Contagem por categoria (RANDOM FOREST)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = {
    "Baixo": "#2ecc71",
    "Médio": "#f39c12",
    "Alto": "#e74c3c",
    "Crítico": "#8e0000"
}

fig, ax = plt.subplots(figsize=(9, 5)) 

# Valores das categorias
vals = [contagem_rf.get(c, 0) for c in ORDEM]

bars = ax.bar(
    ORDEM,
    vals,
    color=[cores[c] for c in ORDEM],
    edgecolor="white",
    width=0.6
)

# Labels (nº + %)
for bar, cat in zip(bars, ORDEM):
    n   = contagem_rf.get(cat, 0)
    pct = percentagem_rf.get(cat, 0.0)
    
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Número de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco (Random Forest)")

plt.tight_layout()
plt.savefig("categorias_risco_rf.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 103.3 Boxplot por categoria (RANDOM FOREST)

import matplotlib.pyplot as plt
import seaborn as sns

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

cores = ["#2ecc71", "#f39c12", "#e74c3c", "#8e0000"]

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df_risco_rf,
    x="nivel_risco",
    y="prob_saida",
    order=ORDEM,
    palette=cores,
    ax=ax
)

ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saída")
ax.set_title("Distribuição das Probabilidades por Categoria (Random Forest)")

plt.tight_layout()
plt.savefig("boxplot_risco_rf.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 104. RESUMO FINAL (RANDOM FOREST)

print("=" * 55)
print("RESUMO — RANDOM FOREST")
print("=" * 55)

print(f"  Modelo:         Random Forest (100 estimadores)")
print(f"  Colaboradores:  {len(df_risco_rf)}")

print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"  {nome:<12}  {resultados_treino_rf[metrica]:>8.4f}  {resultados_teste_rf[metrica]:>8.4f}")

print("\n  Distribuição das categorias de risco:")

print(f"  Baixo:    prob < 30%        -> {contagem_rf.get('Baixo', 0)} colaboradores")
print(f"  Médio:    30% ≤ prob < 50%  -> {contagem_rf.get('Médio', 0)} colaboradores")
print(f"  Alto:     50% ≤ prob < 70%  -> {contagem_rf.get('Alto', 0)} colaboradores")
print(f"  Crítico:  prob ≥ 70%        -> {contagem_rf.get('Crítico', 0)} colaboradores")

print("=" * 55)

# Melhor Modelo

In [ ]:
# 105. TABELA COMPARATIVA 

# 1. GERAR A TABELA COMPARATIVA
tabela_comparativa = pd.DataFrame([
    {"Modelo": "Baseline — Regressão Logística",
     "F1 Treino":        f1_score(y_train,        pipeline.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline.predict_proba(X_test)[:, 1])},

    {"Modelo": "1. Naive Bayes",
     "F1 Treino":        f1_score(y_train,        pipeline_nb.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline_nb.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline_nb.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline_nb.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline_nb.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline_nb.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline_nb.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline_nb.predict_proba(X_test)[:, 1])},

    {"Modelo": "2. LDA",
     "F1 Treino":        f1_score(y_train,        pipeline_lda.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline_lda.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline_lda.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline_lda.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline_lda.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline_lda.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline_lda.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline_lda.predict_proba(X_test)[:, 1])},

    {"Modelo": "3. KNN",
     "F1 Treino":        f1_score(y_train,        pipeline_knn.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline_knn.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline_knn.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline_knn.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline_knn.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline_knn.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline_knn.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline_knn.predict_proba(X_test)[:, 1])},

    {"Modelo": "4. Extra Trees",
     "F1 Treino":        f1_score(y_train,        pipeline_et.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline_et.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline_et.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline_et.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline_et.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline_et.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline_et.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline_et.predict_proba(X_test)[:, 1])},

    {"Modelo": "5. SVM",
     "F1 Treino":        f1_score(y_train,        pipeline_svm.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline_svm.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline_svm.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline_svm.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline_svm.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline_svm.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline_svm.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline_svm.predict_proba(X_test)[:, 1])},

    {"Modelo": "6. AdaBoost",
     "F1 Treino":        f1_score(y_train,        pipeline_ada.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline_ada.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline_ada.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline_ada.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline_ada.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline_ada.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline_ada.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline_ada.predict_proba(X_test)[:, 1])},

    {"Modelo": "7. LightGBM",
     "F1 Treino":        f1_score(y_train,        pipeline_lgb.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline_lgb.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline_lgb.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline_lgb.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline_lgb.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline_lgb.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline_lgb.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline_lgb.predict_proba(X_test)[:, 1])},

    {"Modelo": "8. XGBoost",
     "F1 Treino":        f1_score(y_train,        pipeline_xgb.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline_xgb.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline_xgb.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline_xgb.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline_xgb.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline_xgb.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline_xgb.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline_xgb.predict_proba(X_test)[:, 1])},

    {"Modelo": "9. Random Forest",
     "F1 Treino":        f1_score(y_train,        pipeline_rf.predict(X_train)),
     "Precision Treino": precision_score(y_train, pipeline_rf.predict(X_train)),
     "Recall Treino":    recall_score(y_train,    pipeline_rf.predict(X_train)),
     "AUC Treino":       roc_auc_score(y_train,   pipeline_rf.predict_proba(X_train)[:, 1]),
     "F1 Teste":         f1_score(y_test,         pipeline_rf.predict(X_test)),
     "Precision Teste":  precision_score(y_test,  pipeline_rf.predict(X_test)),
     "Recall Teste":     recall_score(y_test,     pipeline_rf.predict(X_test)),
     "AUC Teste":        roc_auc_score(y_test,    pipeline_rf.predict_proba(X_test)[:, 1])}
]).round(4)

# Coluna de overfitting 
tabela_comparativa["Overfitting (F1)"] = (
    tabela_comparativa["F1 Treino"] - tabela_comparativa["F1 Teste"]
).round(4).apply(lambda x: f"{x:+.4f} " if x > 0.10 else f"{x:+.4f} ✓")

# Ordenar pelo F1 de Teste 
tabela_comparativa = tabela_comparativa.sort_values("F1 Teste", ascending=False).reset_index(drop=True)

print("===== TABELA COMPARATIVA — TODOS OS MODELOS =====")
display(tabela_comparativa)

# O modelo vencedor
melhor = tabela_comparativa.iloc[0]
print(f"\n Modelo vencedor: {melhor['Modelo']}")
print(f"  F1 Treino:         {melhor['F1 Treino']:.4f}")
print(f"  F1 Teste:          {melhor['F1 Teste']:.4f}")
print(f"  AUC-ROC Teste:     {melhor['AUC Teste']:.4f}")
print(f"  Overfitting (F1):  {melhor['Overfitting (F1)']}")

### preciso de tirar normalizações desnecessárias

## Modelo escolhido para Tuning: Regressão Logística

# Rever e melhorar se possível

In [ ]:
# 106. TUNING — REGRESSÃO LOGÍSTICA

# Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])

# Grelha de hiperparâmetros
param_grid_lr = [
    {
        "model__solver": ["liblinear"],
        "model__penalty": ["l1", "l2"],
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__class_weight": [None, "balanced"],
        "model__max_iter": [1000]
    },
    {
        "model__solver": ["lbfgs"],
        "model__penalty": ["l2"],
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__class_weight": [None, "balanced"],
        "model__max_iter": [1000]
    }
]

# Validação cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search
grid_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

# Ajustar apenas no treino
grid_lr.fit(X_train, y_train)

print("===== MELHOR MODELO — REGRESSÃO LOGÍSTICA =====")
print("Melhores parâmetros:")
print(grid_lr.best_params_)

print(f"\nMelhor F1-score médio em CV: {grid_lr.best_score_:.4f}")

In [ ]:
# 107. AVALIAÇÃO DO MODELO OTIMIZADO

best_model = grid_lr.best_estimator_

# Avaliar treino
r_treino_opt = avaliar_modelo(best_model, X_train, y_train, "Treino")

# Avaliar teste
r_teste_opt = avaliar_modelo(best_model, X_test, y_test, "Teste")

In [ ]:
print("===== COMPARAÇÃO: BASELINE vs MODELO OTIMIZADO =====")

for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    
    base = r_teste[metrica]
    opt = r_teste_opt[metrica]
    diff = opt - base
    
    nome = metrica.upper() if metrica != "acc" else "Accuracy"
    
    print(f"{nome:<12} Baseline: {base:.4f} | Otimizado: {opt:.4f} | Dif: {diff:+.4f}")

In [ ]:
r_treino_opt
r_teste_opt

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, recall_score, precision_score

# Probabilidades no conjunto de teste
y_proba_test = best_model.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.1, 0.9, 50)

resultados = []

for t in thresholds:
    y_pred_t = (y_proba_test >= t).astype(int)
    
    resultados.append({
        "threshold": t,
        "f1": f1_score(y_test, y_pred_t),
        "recall": recall_score(y_test, y_pred_t),
        "precision": precision_score(y_test, y_pred_t)
    })

import pandas as pd
df_thresh = pd.DataFrame(resultados)

# Melhor threshold por F1
best_row = df_thresh.loc[df_thresh["f1"].idxmax()]

print("===== MELHOR THRESHOLD =====")
print(best_row)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(df_thresh["threshold"], df_thresh["f1"], label="F1")
plt.plot(df_thresh["threshold"], df_thresh["recall"], label="Recall")
plt.plot(df_thresh["threshold"], df_thresh["precision"], label="Precision")

plt.axvline(best_row["threshold"], color="red", linestyle="--", label="Melhor threshold")

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Otimização do Threshold")
plt.legend()
plt.show()

In [ ]:
# 17. APLICAÇÃO DO THRESHOLD OTIMIZADO

best_threshold = best_row["threshold"]

y_proba_test = best_model.predict_proba(X_test)[:, 1]
y_pred_opt = (y_proba_test >= best_threshold).astype(int)

from sklearn.metrics import classification_report

print("===== MODELO COM THRESHOLD OTIMIZADO =====")
print(f"Threshold utilizado: {best_threshold:.3f}\n")

print(classification_report(y_test, y_pred_opt))

In [ ]:
# 18. MÉTRICAS FINAIS COM THRESHOLD OTIMIZADO

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

best_threshold = float(best_row["threshold"])

# Probabilidades no treino e teste
y_proba_train_opt = best_model.predict_proba(X_train)[:, 1]
y_proba_test_opt  = best_model.predict_proba(X_test)[:, 1]

# Novas previsões com threshold otimizado
y_pred_train_thr = (y_proba_train_opt >= best_threshold).astype(int)
y_pred_test_thr  = (y_proba_test_opt >= best_threshold).astype(int)

# Métricas treino
r_treino_thr = {
    "acc": accuracy_score(y_train, y_pred_train_thr),
    "precision": precision_score(y_train, y_pred_train_thr),
    "recall": recall_score(y_train, y_pred_train_thr),
    "f1": f1_score(y_train, y_pred_train_thr),
    "auc": roc_auc_score(y_train, y_proba_train_opt)
}

# Métricas teste
r_teste_thr = {
    "acc": accuracy_score(y_test, y_pred_test_thr),
    "precision": precision_score(y_test, y_pred_test_thr),
    "recall": recall_score(y_test, y_pred_test_thr),
    "f1": f1_score(y_test, y_pred_test_thr),
    "auc": roc_auc_score(y_test, y_proba_test_opt)
}

print("===== MODELO OTIMIZADO + THRESHOLD OTIMIZADO =====")
print(f"Threshold ótimo: {best_threshold:.4f}\n")

print(f"{'Métrica':<12} {'Treino':>8} {'Teste':>8}")
print("-" * 32)
for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"{nome:<12} {r_treino_thr[metrica]:>8.4f} {r_teste_thr[metrica]:>8.4f}")

print("\n===== CLASSIFICATION REPORT — TESTE =====")
print(classification_report(y_test, y_pred_test_thr, target_names=["Permaneceu", "Saiu"]))

In [ ]:
# 22. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob < 0.50:
        return "Médio"
    elif prob < 0.70:
        return "Alto"
    else:
        return "Crítico"

df_risco["nivel_risco"] = df_risco["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Médio", "Alto", "Crítico"]

contagem = df_risco["nivel_risco"].value_counts()
percentagem = df_risco["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUIÇÃO DAS CATEGORIAS DE RISCO =====")
print(f"{'Categoria':<10} {'Contagem':>8} {'Percentagem':>12}")
print("-" * 36)

for cat in ORDEM:
    n = contagem.get(cat, 0)
    pct = percentagem.get(cat, 0.0)
    print(f"{cat:<10} {n:>8} {pct:>11.1f}%")

In [ ]:
print("===== COMPARAÇÃO FINAL =====")
print(f"{'Métrica':<12} {'Baseline':>10} {'Tuning':>10} {'Tuning+Thr':>12}")
print("-" * 46)

for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(
        f"{nome:<12} "
        f"{r_teste[metrica]:>10.4f} "
        f"{r_teste_opt[metrica]:>10.4f} "
        f"{r_teste_thr[metrica]:>12.4f}"
    )

In [ ]:
# 23. RESUMO FINAL — MODELO FINAL

print("=" * 60)
print("RESUMO — MODELO FINAL")
print("=" * 60)

print(f"Modelo:          Regressão Logística Otimizada")
print(f"Threshold final: {best_threshold:.4f}")
print(f"Colaboradores:   {len(df_risco)}")

print(f"\n{'Métrica':<12} {'Treino':>8} {'Teste':>8}")
print("-" * 32)
for metrica, nome in [
    ("acc", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1-Score"),
    ("auc", "AUC-ROC")
]:
    print(f"{nome:<12} {r_treino_thr[metrica]:>8.4f} {r_teste_thr[metrica]:>8.4f}")

print("\nDistribuição das categorias de risco:")
print(f"Baixo:    prob < 30%        -> {contagem.get('Baixo', 0)} colaboradores")
print(f"Médio:    30% ≤ prob < 50%  -> {contagem.get('Médio', 0)} colaboradores")
print(f"Alto:     50% ≤ prob < 70%  -> {contagem.get('Alto', 0)} colaboradores")
print(f"Crítico:  prob ≥ 70%        -> {contagem.get('Crítico', 0)} colaboradores")

print("=" * 60)

In [ ]:
# 24. TOP 20 COLABORADORES COM MAIOR RISCO

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco.columns:
        cols_top.append(col)

top20 = df_risco.sort_values("prob_saida", ascending=False)[cols_top].head(20)

print("===== TOP 20 — MAIOR PROBABILIDADE DE SAÍDA =====")
display(top20)